In [4]:
import chromadb
import numpy as np

from rank_bm25 import BM25Okapi
import google.generativeai as genai

C:\Users\DELL\AppData\Local\Temp\ipykernel_4848\949770592.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [6]:
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection("azure_collection")
data = collection.get()
documents = data["documents"]

In [7]:
tokenized_docs = [doc.lower().split() for doc in documents]

bm25 = BM25Okapi(tokenized_docs)

In [8]:
def dense_search(query, collection, k=5):
    
    return collection.query(
        query_texts=[query],
        n_results=k
    )

In [ ]:
query = "How do I deploy a Python web application?"
dense_results = dense_search(query, collection)
print(dense_results["documents"][0][0][:300])


Read in English Edit Azure for Python Developers You can deploy your Python applications to Azure in various environments, including web apps, serverless functions, containers, and machine learning models. Azure offers flexible hosting options to suit different architectural and scalability needs. T


In [10]:
def sparse_search(query, bm25, documents, k=5):
    # Calculate BM25 scores
    scores = bm25.get_scores(query.lower().split())

    # Get indices of top k documents
    top_indices = np.argsort(scores)[::-1][:k]

    # Return the corresponding documents
    return [documents[i] for i in top_indices]

In [11]:
query = "How do I deploy a Python web application?"

sparse_results = sparse_search(query, bm25, documents)

for i, doc in enumerate(sparse_results, 1):
    print(f"Rank {i}")
    print(doc[:200])
    print("-"*60)

Rank 1
. Azure libraries (SDK) Get started Get started Get to know the Azure libraries Learn library usage patterns Authenticate with Azure services Web apps Tutorial Quickly create and deploy new Django / F
------------------------------------------------------------
Rank 2
Read in English Edit Note Access to this page requires authorization. You can try signing in or changing directories . Access to this page requires authorization. You can try changing directories . Az
------------------------------------------------------------
Rank 3
. Frequently Asked Questions Q . What happens to my dedicated host in case of a live migration? A . As of today, Azure dedicated hosts do not support live migration and in case of a hardware failure, 
------------------------------------------------------------
Rank 4
. Then you can change the max price in the portal, from the Configuration section for the VM. Q: Can I convert existing VMs to Spot VMs? A: No, setting the Spot flag is only supported at

In [12]:
def hybrid_search(query, collection, bm25, documents, k=5):
    
    dense = dense_search(query, collection, k)["documents"][0]
    sparse = sparse_search(query, bm25, documents, k)

    scores = {}

    # Dense scores
    for rank, doc in enumerate(dense):
        scores[doc] = scores.get(doc, 0) + (k - rank)

    # Sparse scores
    for rank, doc in enumerate(sparse):
        scores[doc] = scores.get(doc, 0) + (k - rank)

    # Sort by combined score
    ranked_docs = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked_docs

In [13]:
query = "How do I deploy a Python web application?"

hybrid_results = hybrid_search(
    query,
    collection,
    bm25,
    documents
)

for i, (doc, score) in enumerate(hybrid_results, 1):
    print(f"Rank {i} | Score: {score}")
    print(doc[:200])
    print("-"*70)

Rank 1 | Score: 6
Read in English Edit Azure for Python Developers You can deploy your Python applications to Azure in various environments, including web apps, serverless functions, containers, and machine learning mo
----------------------------------------------------------------------
Rank 2 | Score: 5
. Azure libraries (SDK) Get started Get started Get to know the Azure libraries Learn library usage patterns Authenticate with Azure services Web apps Tutorial Quickly create and deploy new Django / F
----------------------------------------------------------------------
Rank 3 | Score: 4
Publish packages Tutorial Publish & download artifacts Publish NuGet packages Publish npm packages Publish Python packages Publish and download Universal Packages Publish Maven Packages Publish symbol
----------------------------------------------------------------------
Rank 4 | Score: 4
Read in English Edit Note Access to this page requires authorization. You can try signing in or changing directo

In [15]:
def hybrid_search_weighted(query, collection, bm25, documents, alpha=0.6, k=5):
    
    # Dense retrieval
    dense = dense_search(query, collection, k)["documents"][0]

    # Sparse retrieval
    sparse = sparse_search(query, bm25, documents, k)

    scores = {}

    # Dense contribution
    for rank, doc in enumerate(dense):
        dense_score = (k - rank) / k      # Normalize to [0,1]

        if doc not in scores:
            scores[doc] = 0

        scores[doc] += alpha * dense_score

    # Sparse contribution
    for rank, doc in enumerate(sparse):
        sparse_score = (k - rank) / k

        if doc not in scores:
            scores[doc] = 0

        scores[doc] += (1 - alpha) * sparse_score

    # Sort by final score
    ranked_docs = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked_docs

In [16]:
query = "How do I deploy a Python web application?"

results = hybrid_search_weighted(
    query,
    collection,
    bm25,
    documents,
    alpha=0.6,
    k=5
)

for i, (doc, score) in enumerate(results, 1):
    print(f"Rank {i} | Score = {score:.2f}")
    print(doc[:200])
    print("-" * 70)

Rank 1 | Score = 0.68
Read in English Edit Azure for Python Developers You can deploy your Python applications to Azure in various environments, including web apps, serverless functions, containers, and machine learning mo
----------------------------------------------------------------------
Rank 2 | Score = 0.48
Publish packages Tutorial Publish & download artifacts Publish NuGet packages Publish npm packages Publish Python packages Publish and download Universal Packages Publish Maven Packages Publish symbol
----------------------------------------------------------------------
Rank 3 | Score = 0.40
. Azure libraries (SDK) Get started Get started Get to know the Azure libraries Learn library usage patterns Authenticate with Azure services Web apps Tutorial Quickly create and deploy new Django / F
----------------------------------------------------------------------
Rank 4 | Score = 0.36
Deploy models Deploy Streamline model deployment with endpoints Real-time scoring with online en

In [18]:
queries = [
    "How do I deploy a Python web application?",
    "How do I create a Blob Storage container?",
    "How do I authenticate users with Microsoft Entra ID?"
]

for query in queries:

    print("=" * 100)
    print(f"QUERY: {query}")

    # ---------------- Dense Retrieval ----------------
    print("\nDENSE RETRIEVAL")
    dense_results = dense_search(query, collection)

    for i, doc in enumerate(dense_results["documents"][0], 1):
        print(f"\nRank {i}")
        print(doc[:150])

    # ---------------- Sparse Retrieval ----------------
    print("\nSPARSE RETRIEVAL")
    sparse_results = sparse_search(query, bm25, documents)

    for i, doc in enumerate(sparse_results, 1):
        print(f"\nRank {i}")
        print(doc[:150])

    # ---------------- Hybrid Retrieval ----------------
    print("\nHYBRID RETRIEVAL (Weighted Fusion)")
    hybrid_results = hybrid_search_weighted(
        query,
        collection,
        bm25,
        documents,
        alpha=0.6,
        k=5
    )

    for i, (doc, score) in enumerate(hybrid_results, 1):
        print(f"\nRank {i} | Score: {score:.2f}")
        print(doc[:150])

    print("\n" + "=" * 100 + "\n")

QUERY: How do I deploy a Python web application?

DENSE RETRIEVAL

Rank 1
Read in English Edit Azure for Python Developers You can deploy your Python applications to Azure in various environments, including web apps, serverl

Rank 2
Publish packages Tutorial Publish & download artifacts Publish NuGet packages Publish npm packages Publish Python packages Publish and download Univer

Rank 3
Deploy models Deploy Streamline model deployment with endpoints Real-time scoring with online endpoints Manage the ML lifecycle (MLOps) How-To Guide T

Rank 4
Samples and templates Deploy Azure Lighthouse samples repository (GitHub) Index of Azure Lighthouse samples Work with Azure services at scale How-To G

Rank 5
server Additional Resources Concept Connect with Azure CLI Connection libraries for App development Deploy with Terraform Troubleshooting guides overv

SPARSE RETRIEVAL

Rank 1
. Azure libraries (SDK) Get started Get started Get to know the Azure libraries Learn library usage patterns Auth